<div align="center">

# Heavy Equipment Selling Price Prediction

### Notebook v21: Breakthrough Bug-Fix & Deep Architecture Revival

---

**Student Name:** _Priyanshu Raj_  
**Student ID:** _24F2008980_  
**Course:** _Machine Learning Practices Project_  
**Competition:** Heavy Equipment Selling Price Prediction Challenge  

---

</div>


## Table of Contents

1. [Problem Statement & Breakthrough Strategy](#1)
2. [Dataset Overview](#2)
3. [Import Libraries](#3)
4. [Load Dataset](#4)
5. [Exploratory Data Analysis](#5)
6. [Data Preprocessing & Cleaning](#6)
7. [Feature Engineering: Corrected Regex, Machine Type & Domain Features](#7)
8. [Multi-Percentile OOF Target Encodings & AssetID Price Lags](#8)
9. [Level-1 Base Models Training (10-Fold CV with Early Stopping)](#9)
10. [Level-2 Meta-Learner Stacking & Nelder-Mead Optimization](#10)
11. [Model Performance Comparison & Feature Importance](#11)
12. [Final Predictions & Distribution Verification](#12)
13. [Conclusion & Key Insights](#13)


<a id="1"></a>
# 1. Problem Statement & Breakthrough Strategy

The goal is to predict heavy equipment auction prices (`TargetValue`). The metric is **Root Mean Squared Logarithmic Error (RMSLE)**, calculated as **RMSE on `log1p(TargetValue)`**.

In Notebook v21, we implement a **Breakthrough Architecture** with 5 critical bug fixes identified across v12-v20:

1. **REGEX FIX**: FunctionalClassification capacity/horsepower extraction was broken by double-escaped regex (producing all-NaN features). Now correctly extracts **71K Metric Ton** and **55K Horsepower** values.
2. **MODEL RESTORATION**: Reverted to v12-level deep hyperparameters (LightGBM: 5000 trees, lr=0.012, 140 leaves) with **proper early stopping**. Removed dead-weight linear models (Ridge/Lasso/ElasticNet at 0.28 RMSLE).
3. **AssetID PRICE LAG**: Restored OOF historical price signal (**0.84 correlation** for repeat assets, 3,215 test overlaps).
4. **col1..col30 FIX**: These are categorical strings (equipment options), not floats. Now properly ordinal-encoded.
5. **MACHINE TYPE**: Extract equipment category from FunctionalClassification (6 classes: Excavator, Loader, Dozer, Grader, Backhoe, SSL).


<a id="2"></a>
# 2. Dataset Overview

- **Train Set:** 138,701 rows × 50 columns.
- **Test Set:** 15,000 rows × 49 columns.
- **Target Variable:** `TargetValue` (evaluated on $\\log1p$ transformation).
- **Key Signals:** ManufactureYear, OperationalHoursMeter (40% null), FunctionalClassification (capacity/HP), Spec descriptors, AssetID (repeat-sale history).


<a id="3"></a>
# 3. Import Libraries


In [ ]:
import os
import re
import time
import warnings
from scipy.optimize import minimize

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import RidgeCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print("✅ All v21 Breakthrough libraries successfully imported!")


<a id="4"></a>
# 4. Load Dataset


In [ ]:
data_dir = "."
target_dir_root = "/kaggle/input"

if os.path.exists(target_dir_root):
    for dirpath, _, filenames in os.walk(target_dir_root):
        if "train.csv" in filenames:
            data_dir = dirpath
            break
elif os.path.exists("data"):
    data_dir = "data"
elif os.path.exists("e:/HeavyEquipmentML/data"):
    data_dir = "e:/HeavyEquipmentML/data"

print(f"📂 Data Directory Path: {data_dir}")

df_train_raw = pd.read_csv(os.path.join(data_dir, "train.csv"), low_memory=False)
df_test_raw = pd.read_csv(os.path.join(data_dir, "test.csv"), low_memory=False)
sub_template = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))

print(f"📊 Raw Train Shape : {df_train_raw.shape[0]:,} rows × {df_train_raw.shape[1]} columns")
print(f"📊 Raw Test Shape  : {df_test_raw.shape[0]:,} rows × {df_test_raw.shape[1]} columns")
print(f"📊 Submission Shape: {sub_template.shape[0]:,} rows × {sub_template.shape[1]} columns")


<a id="5"></a>
# 5. Exploratory Data Analysis (EDA)


In [ ]:
log_target_val = np.log1p(df_train_raw["TargetValue"].clip(lower=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_train_raw["TargetValue"], kde=True, ax=axes[0], color="royalblue", bins=50)
axes[0].set_title(f"Original Target Value (Skew: {df_train_raw['TargetValue'].skew():.2f})")

sns.histplot(log_target_val, kde=True, ax=axes[1], color="teal", bins=50)
axes[1].set_title(f"log1p Transformed Target (Skew: {log_target_val.skew():.2f})")
plt.tight_layout()
plt.show()

# Missing value analysis
null_pct = (df_train_raw.isnull().sum() / len(df_train_raw) * 100).sort_values(ascending=False)
print("Top 15 columns by missing percentage:")
print(null_pct[null_pct > 0].head(15).to_string())


<a id="6"></a>
# 6. Data Preprocessing & Cleaning


In [ ]:
def clean_data(train_df, test_df):
    df_tr = train_df.copy()
    df_te = test_df.copy()

    # Drop low-signal columns
    drop_cols = ["InventoryGroupDescription", "TransactionID"]
    df_tr.drop(columns=drop_cols, inplace=True, errors="ignore")
    df_te.drop(columns=drop_cols, inplace=True, errors="ignore")

    # Fix sentinel year 1001 → NaN
    for col in ["YearMade", "ManufactureYear"]:
        if col in df_tr.columns:
            df_tr.loc[df_tr[col] == 1001, col] = np.nan
            df_te.loc[df_te[col] == 1001, col] = np.nan

    # Hours: fix zeros as missing + log transform
    hours_col = "MeterHours" if "MeterHours" in df_tr.columns else "OperationalHoursMeter"
    for df in (df_tr, df_te):
        df["Hours_missing"] = (df[hours_col].isnull() | (df[hours_col] == 0.0)).astype(int)
        df.loc[df[hours_col] == 0.0, hours_col] = np.nan
        df["Hours_log"] = np.log1p(df[hours_col])
        if hours_col != "Hours_log":
            df.drop(columns=[hours_col], inplace=True, errors="ignore")

    # Parse dates
    df_tr["TransactionDate"] = pd.to_datetime(df_tr["TransactionDate"])
    df_te["TransactionDate"] = pd.to_datetime(df_te["TransactionDate"])

    return df_tr, df_te

train_prepped, test_prepped = clean_data(df_train_raw, df_test_raw)
print("✅ Data cleaned successfully!")


<a id="7"></a>
# 7. Feature Engineering: Corrected Regex, Machine Type & Domain Features

> **🔧 CRITICAL FIX**: Previous versions (v7-v20) had double-escaped regex patterns that produced
> all-NaN for `Capacity_Tons_Mid` and `Power_HP_Mid`. This version uses correct single-escaped regex.


In [ ]:
def engineer_features(train_df, test_df):
    df_tr = train_df.copy()
    df_te = test_df.copy()

    mfg_col = "YearMade" if "YearMade" in df_tr.columns else "ManufactureYear"
    col_flag_cols = [c for c in df_tr.columns if c.startswith("col")]

    for dataset in (df_tr, df_te):
        t_date = dataset["TransactionDate"]
        dataset["SaleYear"] = t_date.dt.year
        dataset["SaleMonth"] = t_date.dt.month
        dataset["SaleDay"] = t_date.dt.day
        dataset["SaleDayOfWeek"] = t_date.dt.dayofweek
        dataset["SaleWeek"] = t_date.dt.isocalendar().week.astype(int)
        dataset["SaleQuarter"] = t_date.dt.quarter
        dataset["SaleElapsedDays"] = (t_date - pd.Timestamp("1990-01-01")).dt.days

        year_made = pd.to_numeric(dataset[mfg_col], errors="coerce").fillna(2000)
        dataset["MachineAge"] = (dataset["SaleYear"] - year_made).clip(lower=0)
        dataset["MachineAge_Sq"] = dataset["MachineAge"] ** 2

        # Depreciation & Usage Interactions
        dataset["AgeXHours"] = dataset["MachineAge"] * dataset["Hours_log"]
        dataset["HoursPerYear"] = dataset["Hours_log"] / (dataset["MachineAge"] + 1.0)
        dataset["Depreciation_Curve"] = dataset["MachineAge"] / (np.sqrt(dataset["Hours_log"] + 1.0))

        # ============================================================
        # FIX #1: CORRECT REGEX for FunctionalClassification
        # Previous versions used double-escaped regex (\\d+) that matched NOTHING.
        # ============================================================
        func_str = dataset["FunctionalClassification"].fillna("").astype(str)

        # Extract Metric Tons capacity (covers ~51% of data)
        tons_lo = func_str.str.extract(r"(\d+\.?\d*)\s*to\s*\d+\.?\d*\s*Metric Tons")[0].astype(float)
        tons_hi = func_str.str.extract(r"\d+\.?\d*\s*to\s*(\d+\.?\d*)\s*Metric Tons")[0].astype(float)
        dataset["Capacity_Tons_Mid"] = (tons_lo + tons_hi) / 2.0

        # Extract Horsepower (covers ~40% of data)
        hp_lo = func_str.str.extract(r"(\d+\.?\d*)\s*to\s*\d+\.?\d*\s*Horsepower")[0].astype(float)
        hp_hi = func_str.str.extract(r"\d+\.?\d*\s*to\s*(\d+\.?\d*)\s*Horsepower")[0].astype(float)
        dataset["Power_HP_Mid"] = (hp_lo + hp_hi) / 2.0

        # Extract Digging Depth in Ft (for backhoe loaders)
        depth_lo = func_str.str.extract(r"(\d+\.?\d*)\s*to\s*\d+\.?\d*\s*Ft")[0].astype(float)
        depth_hi = func_str.str.extract(r"\d+\.?\d*\s*to\s*(\d+\.?\d*)\s*Ft")[0].astype(float)
        dataset["Digging_Depth_Mid"] = (depth_lo + depth_hi) / 2.0

        # FIX #5: Extract Machine Type category from FunctionalClassification
        dataset["MachineType"] = func_str.str.extract(r"^(.+?)\s*-\s*")[0].fillna("Unknown")

        # Composite Categorical Tokens
        base_class = dataset["Spec_BaseClass"].astype(str)
        prod_cfg = dataset["ProductConfigID"].astype(str)
        region = dataset["RegionCode"].astype(str) if "RegionCode" in dataset.columns else pd.Series("", index=dataset.index)
        vendor = dataset["VendorPartnerID"].astype(str) if "VendorPartnerID" in dataset.columns else pd.Series("", index=dataset.index)

        dataset["Cat_Prod_Region"] = prod_cfg + "_" + region
        dataset["Cat_Prod_Year"] = prod_cfg + "_" + dataset["SaleYear"].astype(str)
        dataset["Cat_Base_Region"] = base_class + "_" + region

        # Model Token Parsing
        spec_str = dataset["Spec_FullDescriptor"].fillna("").astype(str)
        dataset["ModelToken"] = spec_str.str.extract(r"^([A-Z0-9]+)")[0].fillna("_Missing_")
        dataset["ModelSeriesPrefix"] = spec_str.str.extract(r"^([A-Za-z]+)")[0].fillna("_Missing_")

        # Text Descriptors Metadata
        dataset["Desc_Len"] = spec_str.str.len()
        dataset["Desc_Words"] = spec_str.str.split().str.len()

        # ============================================================
        # FIX #4: col1..col30 are STRINGS (categorical), not numeric.
        # Compute config option count as binary presence count.
        # ============================================================
        dataset["Config_Options_Count"] = dataset[col_flag_cols].notnull().sum(axis=1)

    # Frequency Encodings
    freq_cols = ["Spec_BaseClass", "VendorPartnerID", "ProductConfigID", "ModelToken", "Cat_Prod_Region"]
    for col in freq_cols:
        freq_map = pd.concat([df_tr[col], df_te[col]], axis=0).value_counts().to_dict()
        df_tr[f"{col}_Freq"] = df_tr[col].map(freq_map)
        df_te[f"{col}_Freq"] = df_te[col].map(freq_map)
        df_tr[f"{col}_LogFreq"] = np.log1p(df_tr[f"{col}_Freq"])
        df_te[f"{col}_LogFreq"] = np.log1p(df_te[f"{col}_Freq"])

    # 25-Dimension TF-IDF + TruncatedSVD Text Embeddings
    desc_tr = df_tr["Spec_FullDescriptor"].fillna("").astype(str)
    desc_te = df_te["Spec_FullDescriptor"].fillna("").astype(str)

    tfidf = TfidfVectorizer(max_features=250, stop_words="english", ngram_range=(1, 2))
    svd = TruncatedSVD(n_components=25, random_state=42)

    tfidf_tr = tfidf.fit_transform(desc_tr)
    tfidf_te = tfidf.transform(desc_te)

    svd_tr = svd.fit_transform(tfidf_tr)
    svd_te = svd.transform(tfidf_te)

    for i in range(25):
        df_tr[f"Text_SVD_{i}"] = svd_tr[:, i]
        df_te[f"Text_SVD_{i}"] = svd_te[:, i]

    return df_tr, df_te

df_tr_fe, df_te_fe = engineer_features(train_prepped, test_prepped)

# Verify FunctionalClassification extraction (CRITICAL CHECK)
tons_fill = df_tr_fe["Capacity_Tons_Mid"].notna().sum()
hp_fill = df_tr_fe["Power_HP_Mid"].notna().sum()
print(f"✅ Feature Engineering Complete! Train {df_tr_fe.shape}, Test {df_te_fe.shape}")
print(f"🔧 REGEX FIX VERIFIED: Capacity_Tons_Mid filled: {tons_fill:,} rows ({100*tons_fill/len(df_tr_fe):.1f}%)")
print(f"🔧 REGEX FIX VERIFIED: Power_HP_Mid filled: {hp_fill:,} rows ({100*hp_fill/len(df_tr_fe):.1f}%)")
print(f"🔧 MachineType categories: {df_tr_fe['MachineType'].nunique()}")


<a id="8"></a>
# 8. Multi-Percentile OOF Target Encodings & AssetID Historical Price Lags

> **🔧 FIX #3**: Restored AssetID OOF historical price lag (0.84 correlation for repeat assets).
> 3,215 test AssetIDs overlap with training data.


In [ ]:
TARGET_ENCODE_VARS = [
    "Spec_BaseClass", "Spec_SubClass", "RegionCode", "VendorPartnerID",
    "ProductConfigID", "ModelToken", "Cat_Prod_Region", "Cat_Base_Region",
    "MachineType"
]

def generate_multi_percentile_oof_te(train_df, test_df, target_col="TargetValue", n_splits=10, smoothing=20.0):
    tr = train_df.copy()
    te = test_df.copy()

    y_log = np.log1p(tr[target_col].clip(lower=0))
    g_mean = y_log.mean()
    g_median = y_log.median()
    g_q25 = y_log.quantile(0.25)
    g_q75 = y_log.quantile(0.75)

    bins = pd.qcut(y_log, q=10, labels=False, duplicates="drop")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # ================================================================
    # FIX #3: AssetID OOF Historical Price Lag (0.84 correlation)
    # ================================================================
    if "AssetID" in tr.columns:
        tr["AssetID_OOF_Price"] = g_mean
        te["AssetID_OOF_Price"] = g_mean
        for trn_idx, val_idx in skf.split(tr, bins):
            asset_map = tr.iloc[trn_idx].groupby("AssetID")[target_col].apply(
                lambda s: np.log1p(s.clip(lower=0)).mean()
            ).to_dict()
            tr.iloc[val_idx, tr.columns.get_loc("AssetID_OOF_Price")] = (
                tr.iloc[val_idx]["AssetID"].map(asset_map).fillna(g_mean)
            )
        # Full train mapping for test set
        full_asset_map = tr.groupby("AssetID")[target_col].apply(
            lambda s: np.log1p(s.clip(lower=0)).mean()
        ).to_dict()
        te["AssetID_OOF_Price"] = te["AssetID"].map(full_asset_map).fillna(g_mean)

        overlap_count = te["AssetID"].isin(set(full_asset_map.keys())).sum()
        print(f"🔧 AssetID Price Lag: {overlap_count:,} test rows have historical price data")

    # Multi-Percentile Target Encodings for categorical columns
    for col in TARGET_ENCODE_VARS:
        if col not in tr.columns:
            continue

        m_col = f"{col}_TE_Mean"
        med_col = f"{col}_TE_Median"
        q25_col = f"{col}_TE_Q25"
        q75_col = f"{col}_TE_Q75"

        tr[m_col], tr[med_col], tr[q25_col], tr[q75_col] = g_mean, g_median, g_q25, g_q75
        te[m_col], te[med_col], te[q25_col], te[q75_col] = g_mean, g_median, g_q25, g_q75

        for trn_idx, val_idx in skf.split(tr, bins):
            tr_sub = tr.iloc[trn_idx].copy()
            val_sub = tr.iloc[val_idx].copy()

            tr_sub["target_log"] = np.log1p(tr_sub[target_col].clip(lower=0))
            stats = tr_sub.groupby(col)["target_log"].agg(
                count="count",
                mean="mean",
                median="median",
                q25=lambda x: x.quantile(0.25),
                q75=lambda x: x.quantile(0.75)
            ).reset_index()

            stats["smooth_mean"] = (stats["count"] * stats["mean"] + smoothing * g_mean) / (stats["count"] + smoothing)

            mean_map = stats.set_index(col)["smooth_mean"].to_dict()
            med_map = stats.set_index(col)["median"].to_dict()
            q25_map = stats.set_index(col)["q25"].to_dict()
            q75_map = stats.set_index(col)["q75"].to_dict()

            tr.iloc[val_idx, tr.columns.get_loc(m_col)] = val_sub[col].map(mean_map).fillna(g_mean)
            tr.iloc[val_idx, tr.columns.get_loc(med_col)] = val_sub[col].map(med_map).fillna(g_median)
            tr.iloc[val_idx, tr.columns.get_loc(q25_col)] = val_sub[col].map(q25_map).fillna(g_q25)
            tr.iloc[val_idx, tr.columns.get_loc(q75_col)] = val_sub[col].map(q75_map).fillna(g_q75)

        # Full train mapping for test
        full_sub = tr.copy()
        full_sub["target_log"] = np.log1p(full_sub[target_col].clip(lower=0))
        full_stats = full_sub.groupby(col)["target_log"].agg(
            count="count",
            mean="mean",
            median="median",
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75)
        ).reset_index()

        full_stats["smooth_mean"] = (full_stats["count"] * full_stats["mean"] + smoothing * g_mean) / (full_stats["count"] + smoothing)

        full_mean_map = full_stats.set_index(col)["smooth_mean"].to_dict()
        full_med_map = full_stats.set_index(col)["median"].to_dict()
        full_q25_map = full_stats.set_index(col)["q25"].to_dict()
        full_q75_map = full_stats.set_index(col)["q75"].to_dict()

        te[m_col] = te[col].map(full_mean_map).fillna(g_mean)
        te[med_col] = te[col].map(full_med_map).fillna(g_median)
        te[q25_col] = te[col].map(full_q25_map).fillna(g_q25)
        te[q75_col] = te[col].map(full_q75_map).fillna(g_q75)

    return tr, te

train_te, test_te = generate_multi_percentile_oof_te(df_tr_fe, df_te_fe)
print("✅ Multi-Percentile OOF Target Encodings + AssetID Price Lags Generated!")


<a id="9"></a>
# 9. Level-1 Base Models Training (10-Fold CV with Early Stopping)

> **🔧 FIX #2**: Restored v12-level deep hyperparameters with early stopping.
> Removed dead-weight Ridge/Lasso/ElasticNet (0.28 RMSLE noise).
> Fresh model instantiation per fold.


In [ ]:
# Prepare feature matrix
drop_cols = ["SalesID", "TransactionDate", "TargetValue", "AssetID"]
feature_cols = [c for c in train_te.columns if c not in drop_cols]

X = train_te[feature_cols].copy()
y_log = np.log1p(train_te["TargetValue"].clip(lower=0))
X_test = test_te[feature_cols].copy()

# Ordinal encode all object/category columns (FIX #4: col* are now properly handled)
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
for col in categorical_cols:
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[col] = enc.fit_transform(X[[col]].astype(str))
    X_test[col] = enc.transform(X_test[[col]].astype(str))

X = X.fillna(-999)
X_test = X_test.fillna(-999)

print(f"📊 Feature Matrix: {X.shape[1]} features")
print(f"📊 Categorical columns encoded: {len(categorical_cols)}")


In [ ]:
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
target_bins = pd.qcut(y_log, q=10, labels=False, duplicates="drop")

oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_et = np.zeros(len(X))

test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_et = np.zeros(len(X_test))

# ================================================================
# FIX #2: v12-Level Deep Hyperparameters with Early Stopping
# ================================================================

# LightGBM — Deep leaf-wise growth (restored from v12)
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.012,
    "num_leaves": 140,
    "max_depth": 11,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_estimators": 5000,
    "verbose": -1,
    "n_jobs": -1
}

# CatBoost — Symmetric tree growth
cat_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "learning_rate": 0.018,
    "depth": 8,
    "l2_leaf_reg": 5,
    "subsample": 0.8,
    "random_seed": 42,
    "iterations": 4500,
    "verbose": False,
    "thread_count": -1
}

# XGBoost — Histogram-based gradient boosting
xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.012,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_estimators": 4500,
    "tree_method": "hist",
    "n_jobs": -1
}

# ExtraTrees — Extremely randomized splits
et_params = {
    "n_estimators": 200,
    "max_depth": 24,
    "min_samples_split": 4,
    "n_jobs": -1,
    "random_state": 42
}

EARLY_STOP = 150

print(f"=== Training Level-1 Base Models ({N_SPLITS}-Fold CV) ===")
print(f"    LGB: {lgb_params['n_estimators']} trees, lr={lgb_params['learning_rate']}, leaves={lgb_params['num_leaves']}")
print(f"    CAT: {cat_params['iterations']} iters, lr={cat_params['learning_rate']}, depth={cat_params['depth']}")
print(f"    XGB: {xgb_params['n_estimators']} trees, lr={xgb_params['learning_rate']}, depth={xgb_params['max_depth']}")
print(f"    ET:  {et_params['n_estimators']} trees, depth={et_params['max_depth']}")
print(f"    Early stopping patience: {EARLY_STOP}")
print()

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, target_bins), start=1):
    f_start = time.time()
    X_tr, y_tr = X.iloc[trn_idx], y_log.iloc[trn_idx]
    X_va, y_va = X.iloc[val_idx], y_log.iloc[val_idx]

    # LightGBM — FRESH model per fold + early stopping
    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict(X_va)
    test_lgb += m_lgb.predict(X_test) / N_SPLITS

    # CatBoost — FRESH model per fold + early stopping
    m_cat = CatBoostRegressor(**cat_params)
    m_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va),
              early_stopping_rounds=EARLY_STOP, verbose=False)
    oof_cat[val_idx] = m_cat.predict(X_va)
    test_cat += m_cat.predict(X_test) / N_SPLITS

    # XGBoost — FRESH model per fold + early stopping
    m_xgb = xgb.XGBRegressor(**xgb_params)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict(X_va)
    test_xgb += m_xgb.predict(X_test) / N_SPLITS

    # ExtraTrees — FRESH model per fold
    m_et = ExtraTreesRegressor(**et_params)
    m_et.fit(X_tr, y_tr)
    oof_et[val_idx] = m_et.predict(X_va)
    test_et += m_et.predict(X_test) / N_SPLITS

    fold_lgb = np.sqrt(mean_squared_error(y_va, oof_lgb[val_idx]))
    print(f"Fold {fold}/{N_SPLITS} — LGB: {fold_lgb:.5f} — {time.time() - f_start:.1f}s")

lgb_rmsle = np.sqrt(mean_squared_error(y_log, oof_lgb))
cat_rmsle = np.sqrt(mean_squared_error(y_log, oof_cat))
xgb_rmsle = np.sqrt(mean_squared_error(y_log, oof_xgb))
et_rmsle = np.sqrt(mean_squared_error(y_log, oof_et))

print()
print(f"✅ LightGBM   10-Fold OOF RMSLE: {lgb_rmsle:.5f}")
print(f"✅ CatBoost   10-Fold OOF RMSLE: {cat_rmsle:.5f}")
print(f"✅ XGBoost    10-Fold OOF RMSLE: {xgb_rmsle:.5f}")
print(f"✅ ExtraTrees 10-Fold OOF RMSLE: {et_rmsle:.5f}")


<a id="10"></a>
# 10. Level-2 Meta-Learner Stacking & Nelder-Mead Optimization


In [ ]:
# Stack OOF predictions
oof_matrix = pd.DataFrame({
    "lgb": oof_lgb,
    "cat": oof_cat,
    "xgb": oof_xgb,
    "et": oof_et
})

test_matrix = pd.DataFrame({
    "lgb": test_lgb,
    "cat": test_cat,
    "xgb": test_xgb,
    "et": test_et
})

# Level-2 Meta Learner: RidgeCV
meta_ridge = RidgeCV(alphas=np.logspace(-3, 3, 20), cv=10)
meta_ridge.fit(oof_matrix, y_log)

meta_oof_ridge = meta_ridge.predict(oof_matrix)
meta_test_ridge = meta_ridge.predict(test_matrix)
score_ridge = np.sqrt(mean_squared_error(y_log, meta_oof_ridge))

print(f"🎯 Level-2 RidgeCV Meta-Learner OOF RMSLE: {score_ridge:.5f}")
print("   Ridge Meta Weights:", dict(zip(oof_matrix.columns, np.round(meta_ridge.coef_, 4))))

# Non-Negative Nelder-Mead Weight Optimization
def loss_func(weights):
    weights = np.abs(weights)
    weights = weights / np.sum(weights)
    blend = np.dot(oof_matrix.values, weights)
    return np.sqrt(mean_squared_error(y_log, blend))

n_models = len(oof_matrix.columns)
init_weights = [1.0 / n_models] * n_models

res = minimize(loss_func, init_weights, method="Nelder-Mead",
               options={"maxiter": 10000, "xatol": 1e-8, "fatol": 1e-8})
opt_weights = np.abs(res.x) / np.sum(np.abs(res.x))

meta_oof_opt = np.dot(oof_matrix.values, opt_weights)
meta_test_opt = np.dot(test_matrix.values, opt_weights)
score_opt = np.sqrt(mean_squared_error(y_log, meta_oof_opt))

print(f"🎯 Nelder-Mead Optimal OOF RMSLE: {score_opt:.5f}")
print("   Optimal Weights:", dict(zip(oof_matrix.columns, np.round(opt_weights, 4))))

# Select best
if score_opt < score_ridge:
    final_oof = meta_oof_opt
    final_test = meta_test_opt
    best_stack_score = score_opt
    best_meta_name = "Nelder-Mead Optimal Blend"
    print("🏆 Selected Nelder-Mead Optimal Stacking Blend!")
else:
    final_oof = meta_oof_ridge
    final_test = meta_test_ridge
    best_stack_score = score_ridge
    best_meta_name = "RidgeCV Meta-Learner"
    print("🏆 Selected RidgeCV Meta-Learner Stacking Blend!")

print(f"\n🔥 FINAL LEVEL-2 META RMSLE: {best_stack_score:.5f} ({best_meta_name})")


<a id="11"></a>
# 11. Model Performance Comparison & Feature Importance


In [ ]:
results_df = pd.DataFrame([
    {"Model": "LightGBM", "OOF RMSLE": lgb_rmsle},
    {"Model": "CatBoost", "OOF RMSLE": cat_rmsle},
    {"Model": "XGBoost", "OOF RMSLE": xgb_rmsle},
    {"Model": "ExtraTrees", "OOF RMSLE": et_rmsle},
    {"Model": f"🔥 {best_meta_name}", "OOF RMSLE": best_stack_score},
]).sort_values(by="OOF RMSLE")

print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=results_df, x="OOF RMSLE", y="Model", palette="viridis")
plt.title("v21 Model OOF RMSLE Comparison (Lower is Better)")
plt.xlim(min(results_df["OOF RMSLE"]) * 0.98, max(results_df["OOF RMSLE"]) * 1.02)
plt.tight_layout()
plt.show()

# Feature Importance (LightGBM)
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": m_lgb.feature_importances_
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=fi.head(30), x="importance", y="feature", palette="magma")
plt.title("Top 30 LightGBM Feature Importances (Last Fold)")
plt.tight_layout()
plt.show()

# Verify that new features rank high
new_features = ["Capacity_Tons_Mid", "Power_HP_Mid", "AssetID_OOF_Price", "MachineType", "Digging_Depth_Mid"]
for f in new_features:
    if f in fi["feature"].values:
        rank = fi.index.get_loc(fi[fi["feature"] == f].index[0]) + 1
        imp = fi[fi["feature"] == f]["importance"].values[0]
        print(f"  🔧 {f}: Rank #{rank}, Importance={imp}")


<a id="12"></a>
# 12. Final Predictions & Distribution Verification


In [ ]:
final_pred_values = np.expm1(final_test)
final_pred_values = np.clip(final_pred_values, a_min=0, a_max=None)

sub = pd.DataFrame({
    "TransactionID": df_test_raw["TransactionID"],
    "TargetValue": final_pred_values
})

sub.to_csv("submission.csv", index=False)
print("✅ submission.csv successfully saved!")
print(f"📊 Submission Shape: {sub.shape}")
print(f"📊 Target Summary Statistics:")
print(sub["TargetValue"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Submission distribution
sns.histplot(np.log1p(sub["TargetValue"]), kde=True, color="purple", bins=50, ax=axes[0])
axes[0].set_title("Distribution of Final Submission (log1p)")

# Compare with train distribution
sns.histplot(np.log1p(df_train_raw["TargetValue"]), kde=True, color="teal", bins=50, ax=axes[1], alpha=0.5, label="Train")
sns.histplot(np.log1p(sub["TargetValue"]), kde=True, color="purple", bins=50, ax=axes[1], alpha=0.5, label="Submission")
axes[1].legend()
axes[1].set_title("Train vs Submission Distribution Overlap")

plt.tight_layout()
plt.show()


<a id="13"></a>
# 13. Conclusion & Key Insights

### Critical Bugs Fixed in v21
1. **Regex Fix**: FunctionalClassification HP/Tons extraction was broken since v7 (double-escaped regex). Now correctly extracts 71K+ capacity and 55K+ horsepower values.
2. **Model Architecture**: Restored v12-level deep hyperparameters (5000 LGB trees, 0.012 lr, 140 leaves) with proper early stopping. Removed 0.28-RMSLE linear model noise.
3. **AssetID Price Signal**: Restored 0.84-correlation historical price lag for repeat assets (3,215 test overlap).
4. **Categorical Fix**: col1..col30 properly handled as categorical strings, not float flags.
5. **Machine Type**: Extracted equipment category from FunctionalClassification text.

### Architecture Summary
- **Level-1**: 4 tree-based models (LightGBM, CatBoost, XGBoost, ExtraTrees) with 10-fold CV + early stopping.
- **Level-2**: RidgeCV + Nelder-Mead non-negative weight optimization.
- **Key Features**: Corrected HP/Tons, AssetID price lag, 25-dim text SVD, multi-percentile OOF target encodings.

